# LLM-as-a-Judge — Grupo 14 (IIC3633)

Evalúa las JUDGE_K muestras de un archivo de predicciones (método × dataset) en tres dimensiones reference-free: **relevance** (título #1 vs. preferencias expresadas), **coherence** (mensaje completo en contexto), **explainability** (justificación del título #1 solamente).

Juez: `gemma4-31b-it`, `temperature=0`, `thinking_level='minimal'`, salida JSON estructurada (Pydantic).

**Requisitos:**
- `utils.py` en la misma carpeta que este notebook.
- API Key de google gemini

## 1. Entorno y API key

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata, drive
    os.environ["GEMINI_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/Proyecto RecSys"
    os.chdir(PROJECT_DIR + "/")
else:
    # Local: exporta la key ANTES de abrir Jupyter:
    #   export GEMINI_API_KEY="tu_key_aqui"
    if not os.environ.get("GEMINI_API_KEY"):
        raise RuntimeError(
            "GEMINI_API_KEY no está seteada. En local: "
            "`export GEMINI_API_KEY=...` en la terminal antes de correr Jupyter."
        )

print("Entorno:", "Colab" if IN_COLAB else "Local")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Entorno: Colab


In [ ]:
!pip install -q -U google-genai pydantic

In [ ]:
import json, time, random
from pydantic import BaseModel, field_validator
from google import genai
from google.genai import types

# utils.py debe estar en la misma carpeta que este notebook
from utils import download_dataset, load_parsed, build_context

## 2. Config — completar manualmente con TUS archivos asignados

In [ ]:
# method: cot | reflexion | dc | ace_base | ace_tools
# dataset: "redial" | "pearl"
# predictions_path: JSON ya generado por el notebook del método (idx, predicted, message, ...)
# dataset_path: .jsonl (redial) o .json (pearl) del test set original
# judge_output_path: donde se guardan los scores del juez (checkpointeable)

RUNS = [
    # Agrega aquí el resto de TUS archivos asignados (deja solo los tuyos activos)
    #pearl
    {
        "method": "ace_base",
        "dataset": "pearl",
        "predictions_path": "ace_v2/pearl/ace_results_base_pearl.json",
        "judge_output_path": "judge/ace_base_pearl_judge.json",
    },
    {
        "method": "ace_tools",
        "dataset": "pearl",
        "predictions_path": "ace_v2/pearl/ace_results_tools_pearl.json",
        "judge_output_path": "judge/ace_tools_pearl_judge.json",
    },
    # redial
    {
        "method": "ace_base",
        "dataset": "redial",
        "predictions_path": "ace_v2/redial/ace_results_base_redial.json",
        "judge_output_path": "judge/ace_base_redial_judge.json",
    },
    {
        "method": "ace_tools",
        "dataset": "redial",
        "predictions_path": "ace_v2/redial/ace_results_tools_redial.json",
        "judge_output_path": "judge/ace_tools_redial_judge.json",
    }
]

JUDGE_MODEL = "gemma-4-31b-it" # "gemini-3-flash-preview"
RATE_LIMIT_SECONDS = 5 # para 15 rpm # 6.5      # ~10 RPM free tier Gemini 3 Flash + margen
CHECKPOINT_EVERY = 20
MAX_RETRIES = 5

## 3. Rúbrica del juez (prompt + schema)

In [ ]:
SYSTEM_PROMPT = """You are an expert evaluator of conversational movie recommender systems. You will be shown a conversation history between a user (seeker) and a recommendation assistant, followed by the assistant's final response: a natural-language message recommending a ranked list of titles.

Score the response on three dimensions, each a 1-5 integer. Use these anchors exactly.

## Relevance (score ONLY the first recommended title, against preferences the user expressed in the conversation)
1 - Contradicts or ignores the user's stated preferences.
2 - Tangentially related; shares at most a superficial attribute, no real connection to what the user asked.
3 - Partially aligned; matches one explicit preference but ignores others mentioned.
4 - Aligned with the user's explicit stated preferences (genre, actor, mood, etc.).
5 - Aligned with explicit preferences AND implicit signals in the conversation (tone, taste pattern, rejected alternatives).

ANTI-ECHO RULE: if the first recommended title was already mentioned earlier in the conversation (by either speaker), this is not a valid new recommendation. Score relevance = 1 regardless of topical fit.

## Coherence (score the assistant's message as a whole, in context)
1 - Irrelevant to the last turn; ignores the conversation.
2 - Responds but breaks the flow.
3 - Coherent but generic; could fit almost any similar conversation.
4 - Clearly contextualized to the user's last turn.
5 - Natural continuation that also picks up on earlier signals, not just the last turn.

## Explainability (score ONLY the justification clause for the first title -- the \"because ...\" part. Ignore the rest of the message.)
1 - No justification given.
2 - Generic justification, not tied to this user (\"it's a great movie\", \"it's popular\").
3 - Plausible reason, not explicitly linked to anything the user said.
4 - Reason explicitly linked to one concrete stated preference.
5 - Reason connects specific movie attributes to explicit preferences the user stated.

Ground truth is never provided and must not be assumed. Judge only from the conversation and response shown below. Provide a one-sentence reasoning per dimension before its score."""

USER_TEMPLATE = """### Conversation history
{context}

### Assistant's message
{message}"""

class JudgeScore(BaseModel):
    relevance_reasoning: str
    relevance_score: int
    coherence_reasoning: str
    coherence_score: int
    explainability_reasoning: str
    explainability_score: int

    @field_validator("relevance_score", "coherence_score", "explainability_score")
    @classmethod
    def check_range(cls, v):
        if not (1 <= v <= 5):
            raise ValueError(f"Score out of range 1-5: {v}")
        return v

## 4. Cliente y función de juicio

In [ ]:
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [ ]:
def judge_one(context: str, message: str, label: str = "") -> dict:
    user_content = USER_TEMPLATE.format(context=context, message=message)
    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            response = client.models.generate_content(
                model=JUDGE_MODEL,
                contents=user_content,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0,
                    thinking_config=types.ThinkingConfig(thinking_level="minimal"),
                    response_mime_type="application/json",
                    response_schema=JudgeScore,
                    max_output_tokens=1024,
                ),
            )
            data = json.loads(response.text)
            parsed = JudgeScore(**data)
            return parsed.model_dump()
        except Exception as e:
            last_err = e
            wait = (2 ** attempt) + random.random()
            print(f"  [{label}] [retry {attempt+1}/{MAX_RETRIES}] {e} -> esperando {wait:.1f}s")
            time.sleep(wait)
    raise RuntimeError(f"[{label}] Se agotaron los reintentos: {last_err}")

## 5. Sanity check — correr ANTES del loop completo

In [ ]:
def get_test_parsed(dataset: str):
    """Descarga (o reutiliza si ya existe localmente) el test set y lo parsea."""
    paths = download_dataset(dataset, splits=("test",))
    return load_parsed(paths["test"], dataset=dataset)

In [ ]:
def sanity_check(run_cfg, n=3):
    preds = json.load(open(run_cfg["predictions_path"]))
    parsed = get_test_parsed(run_cfg["dataset"])
    for p in preds[:n]:
        dialogue = parsed[p["idx"]]
        context = build_context(dialogue)
        result = judge_one(context, p["message"])
        print(f"--- idx {p['idx']} ---")
        print("Context (tail):", context[-300:])
        print("Message:", p["message"])
        print("Scores:", result)
        print()


In [ ]:
# Descomenta y corre esto primero, revisa los 3 resultados manualmente:
for run in RUNS:
  sanity_check(run, n=3)

pearl_test.json ya existe, omitiendo descarga.
--- idx 0 ---
Context (tail): rected by James Foley, known for his skill in handling intense and gritty films.
User: "The Corruptor" sounds like a solid movie, but I'm really looking for something with Bruce Willis' intense performance as the lead, and the intense and gritty tone of the movie. Do you have something else in mind?
Message: I recommend 'The Last Boy Scout' (1991) because it features Bruce Willis in a gritty, intense role with a memorable opening scene and delivers the action-packed, entertaining tone you're looking for. You might also enjoy 'Misery' (1990), 'The Sixth Sense' (1999), 'The Negotiator' (1998), and '12 Monkeys' (1995).
Scores: {'relevance_reasoning': "The recommendation matches the user's request for Bruce Willis, an intense/gritty tone, and a seasoned director (Tony Scott), and it was not previously mentioned.", 'relevance_score': 5, 'coherence_reasoning': "The response directly addresses the user's specific req

## 6. Loop completo con checkpointing

In [ ]:
# def run_judge(run_cfg):
#     method, dataset = run_cfg["method"], run_cfg["dataset"]
#     preds = json.load(open(run_cfg["predictions_path"]))
#     parsed = get_test_parsed(dataset)

#     out_path = run_cfg["judge_output_path"]
#     os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
#     if os.path.exists(out_path):
#         results = json.load(open(out_path))
#         done_idx = {r["idx"] for r in results}
#     else:
#         results = []
#         done_idx = set()

#     print(f"[{method}-{dataset}] {len(done_idx)}/{len(preds)} ya evaluados")

#     for i, p in enumerate(preds, 1):
#         if p["idx"] in done_idx:
#             continue
#         dialogue = parsed[p["idx"]]
#         context = build_context(dialogue)
#         label = f"{method}-{dataset} {i}/{len(preds)} idx={p['idx']}"
#         scores = judge_one(context, p["message"], label=label)
#         scores["idx"] = p["idx"]
#         results.append(scores)

#         if len(results) % CHECKPOINT_EVERY == 0:
#             json.dump(results, open(out_path, "w"), indent=2)
#             print(f"  checkpoint: {len(results)}/{len(preds)}")

#         time.sleep(RATE_LIMIT_SECONDS)

#     json.dump(results, open(out_path, "w"), indent=2)
#     print(f"[{method}-{dataset}] completo: {len(results)} juicios guardados en {out_path}")

In [ ]:
# que maneje fallos y los registre en vez de caerse
def run_judge(run_cfg):
    method, dataset = run_cfg["method"], run_cfg["dataset"]
    preds = json.load(open(run_cfg["predictions_path"]))
    parsed = get_test_parsed(dataset)

    out_path = run_cfg["judge_output_path"]
    failed_path = out_path.replace(".json", "_failed.json")
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)

    results = json.load(open(out_path)) if os.path.exists(out_path) else []
    done_idx = {r["idx"] for r in results}
    failed = json.load(open(failed_path)) if os.path.exists(failed_path) else []

    print(f"[{method}-{dataset}] {len(done_idx)}/{len(preds)} ya evaluados, {len(failed)} fallidos previos (se reintentan)")

    for i, p in enumerate(preds, 1):
        if p["idx"] in done_idx:
            continue
        dialogue = parsed[p["idx"]]
        context = build_context(dialogue)
        label = f"{method}-{dataset} {i}/{len(preds)} idx={p['idx']}"

        try:
            scores = judge_one(context, p["message"], label=label)
            scores["idx"] = p["idx"]
            results.append(scores)
        except Exception as e:
            print(f"  ⚠️ [{label}] FALLÓ definitivamente tras {MAX_RETRIES} intentos, se omite: {e}")
            failed = [f for f in failed if f["idx"] != p["idx"]] + [{"idx": p["idx"], "error": str(e)}]
            json.dump(failed, open(failed_path, "w"), indent=2)
            continue  # <-- esto es lo que faltaba: sigue con el siguiente, no crashea

        if len(results) % CHECKPOINT_EVERY == 0:
            json.dump(results, open(out_path, "w"), indent=2)
            print(f"  checkpoint: {len(results)}/{len(preds)}")

        time.sleep(RATE_LIMIT_SECONDS)

    json.dump(results, open(out_path, "w"), indent=2)
    print(f"[{method}-{dataset}] completo: {len(results)}/{len(preds)} evaluados, {len(failed)} fallidos definitivos -> {out_path}")
    if failed:
        print(f"  idx fallidos: {[f['idx'] for f in failed]} (guardados en {failed_path}, re-corre esta celda para reintentarlos)")

In [ ]:
# RUN

for run_cfg in RUNS:
    run_judge(run_cfg)

pearl_test.json ya existe, omitiendo descarga.
[ace_base-pearl] 0/300 ya evaluados, 0 fallidos previos (se reintentan)
  [ace_base-pearl 1/300 idx=0] [retry 1/5] 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}} -> esperando 1.6s
  [ace_base-pearl 1/300 idx=0] [retry 2/5] 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}} -> esperando 2.5s
  [ace_base-pearl 1/300 idx=0] [retry 3/5] 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}} -> esperando 4.1s
  [ace_base-pearl 1/300 idx=0] [retry 4/5] 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}} -> esperando 8.1s
  [ace_base-pearl 6/300 idx=5] [retry 1/5] 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}} -> esperando 1.0s
  [ace_base-pearl 7/300 idx=6] [retry 1/5] 500 INTERNAL. {'er

## Siguiente paso (fuera de este notebook)

Una vez que los 4 integrantes tengan sus `judge_scores/*.json`, se agregan en un notebook aparte de análisis: concatenar los 10 archivos en un DataFrame long-format, verificar que los `idx` coincidan entre los 5 métodos por dataset (diseño pareado), y correr Friedman + Wilcoxon post-hoc.

------

## Implementación y justificación

- **Juez independiente del generador:** Gemini 3 Flash evalúa outputs de Qwen3-8B. Evita el *self-preference bias* — un modelo que se evalúa a sí mismo se favorece sistemáticamente, y el efecto escala con la capacidad de autorreconocimiento del modelo evaluado [11].
- **Paradigma validado en literatura general:** LLM-as-Judge alcanza >80% de acuerdo con humanos usando un juez fuerte (GPT-4 en el estudio original), nivel comparable al acuerdo inter-humano [10].
- **Diseño pointwise, Likert 1-5, reference-free:** no se expone el ground truth al juez — evita que el juez simplemente reproduzca el sesgo temporal/incompletitud del GT de ReDial ya documentado como limitación propia del proyecto.
- **Diseño pareado:** mismos 300 índices evaluados en los 5 métodos por dataset → permite comparación intra-diálogo (Friedman + Wilcoxon post-hoc) en vez de grupos independientes.
- **Razonamiento explícito antes del score**, forzado vía schema JSON — hace auditable la justificación de cada score (a diferencia de razonamiento interno no observable del modelo).

## Qué mide (y por qué esas 3 dimensiones, no otras)

- **Relevance** (solo título #1): ajuste a preferencias *expresadas en el diálogo*, no al GT — recupera falsos negativos que Recall/MRR penalizan por diseño. Opera el objetivo de *effectiveness* de la taxonomía fundacional de explicación en RecSys [12].
- **Coherence**: ajuste conversacional del mensaje completo al hilo del diálogo.
- **Explainability** (solo la cláusula justificativa del título #1): mide si la razón dada conecta atributos concretos de la película con preferencias explícitas del usuario — operacionaliza directamente el objetivo de *persuasiveness* de [12], la referencia que establece por qué la explicabilidad es un eje de evaluación necesario en sistemas de recomendación, más allá de la precisión del ranking.

## Sesgos y errores posibles, por tipo de modelo/emparejamiento

- **Riesgo evitado por diseño — self-preference:** si el juez y el generador fueran el mismo modelo (Qwen3-8B evaluándose a sí mismo), el sesgo de autopreferencia sería significativo y correlacionaría con la capacidad de autorreconocimiento del modelo [11]. Se evita usando un juez de familia distinta (Gemini vs. Qwen).
- **Riesgo no eliminable — calibración de dominio:** la validación de acuerdo juez-humano en [10] y [11] se hizo sobre chat/instruction-following general, no sobre recomendación conversacional de películas. No se realizó validación humana propia sobre esta instancia específica (limitación explícita del alcance del proyecto, declarada en el paper en vez de asumida).
- **Riesgo estructural del juez único:** sin un segundo juez o validación humana, cualquier error sistemático de Gemini 3 Flash en este dominio específico no sería detectable — mitigado parcialmente por fundamentar las 3 dimensiones en literatura validada [12] en vez de definirlas ad hoc.

## Referencias (agregar a la bibliografía, numeración a ajustar según tu lista actual)

[10] Zheng, L., Chiang, W.-L., Sheng, Y., Zhuang, S., Zhuang, Y., Lin, Z., Li, Z., Li, D., Xing, E. P., Zhang, H., Gonzalez, J. E., & Stoica, I. (2023). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena.* In Advances in Neural Information Processing Systems 36 (NeurIPS 2023), Datasets and Benchmarks Track. arXiv:2306.05685.

[11] Panickssery, A., Bowman, S. R., & Feng, S. (2024). *LLM Evaluators Recognize and Favor Their Own Generations.* In Advances in Neural Information Processing Systems 37 (NeurIPS 2024). arXiv:2404.13076.

[12] Tintarev, N., & Masthoff, J. (2007). *Effective Explanations of Recommendations: User-Centered Design.* In Proceedings of the 2007 ACM Conference on Recommender Systems (RecSys'07), 153–156. https://doi.org/10.1145/1297231.1297259

In [ ]:
# import pandas as pd

# def summarize_all_runs(runs, save_csv="judge/summary_scores.csv"):
#     rows = []
#     for run_cfg in runs:
#         method, dataset = run_cfg["method"], run_cfg["dataset"]
#         path = run_cfg["judge_output_path"]
#         if not os.path.exists(path):
#             print(f"⚠️ No existe {path}, saltando {method}-{dataset}")
#             continue

#         results = json.load(open(path))
#         df = pd.DataFrame(results)
#         n = len(df)

#         row = {
#             "method": method,
#             "dataset": dataset,
#             "n": n,
#             "relevance_mean": df["relevance_score"].mean(),
#             "relevance_std": df["relevance_score"].std(),
#             "coherence_mean": df["coherence_score"].mean(),
#             "coherence_std": df["coherence_score"].std(),
#             "explainability_mean": df["explainability_score"].mean(),
#             "explainability_std": df["explainability_score"].std(),
#         }
#         rows.append(row)

#     summary = pd.DataFrame(rows).sort_values(["dataset", "method"]).reset_index(drop=True)
#     os.makedirs(os.path.dirname(save_csv) or ".", exist_ok=True)
#     summary.to_csv(save_csv, index=False)
#     print(f"Guardado en {save_csv}")
#     return summary



In [ ]:
# def worst_cases(run_cfg, metric="relevance_score", n=3):
#     results = json.load(open(run_cfg["judge_output_path"]))
#     df = pd.DataFrame(results).sort_values(metric).head(n)
#     return df[["idx", metric, metric.replace("_score", "_reasoning")]]

# worst_cases(RUNS[0], metric="relevance_score", n=3)

In [ ]:
# def tag_failure_patterns(run_cfg, metric="relevance_score", threshold=2):
#     df = pd.DataFrame(json.load(open(run_cfg["judge_output_path"])))
#     low = df[df[metric] <= threshold]
#     reasoning_col = metric.replace("_score", "_reasoning")
#     patterns = {
#         "echo": low[reasoning_col].str.contains("ECHO|earlier|already mentioned", case=False, na=False).sum(),
#         "ignora_preferencia": low[reasoning_col].str.contains("ignores|but the assistant recommended", case=False, na=False).sum(),
#         "justificacion_generica": low[reasoning_col].str.contains("generic|not tied|not explicitly", case=False, na=False).sum(),
#     }
#     return patterns, low

In [ ]:
# summary_df = summarize_all_runs(RUNS)
# summary_df

In [ ]:
# pivot = summary_df.pivot(index="method", columns="dataset",
#                           values=["relevance_mean", "coherence_mean", "explainability_mean"])
# pivot

In [ ]:
# run_lookup = {(r["method"], r["dataset"]): r for r in RUNS}